# 1. Install packages

In [1]:
!pip install transformers datasets scikit-learn pandas --quiet

import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import torch
import os

import warnings
warnings.filterwarnings('ignore')

# 2. Mount dataset

In [2]:
# First, check what data is available
print("Available datasets:")
!ls /kaggle/input/

DATA_PATH = '/kaggle/input/datasets/fokase/customer-review/data.txt'

# Check if file exists
if not os.path.exists(DATA_PATH):
    print(f"File not found at {DATA_PATH}")
    print("Listing available files:")
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            print(os.path.join(root, file))
    # Use the correct path
    DATA_PATH = '/kaggle/input/sentiment-analysis-data/data.txt'

Available datasets:
datasets


# 3. Load FastText data

In [3]:
def load_fasttext_data(path):
    texts = []
    labels = []
    
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith("__label__"):
                parts = line.split(" ", 1)
                label = parts[0]
                text = parts[1] if len(parts) > 1 else ""
                
                texts.append(text)
                labels.append(label)
    
    df = pd.DataFrame({
        "text": texts,
        "raw_label": labels
    })
    
    return df

print("Loading data...")
df_full = load_fasttext_data(DATA_PATH)
print(f"Full dataset size: {len(df_full)}")

Loading data...
Full dataset size: 400000


# 4. Map labels to sentiment

In [4]:
def map_fasttext_label(label):
    if label == "__label__2":
        return "positive"
    elif label == "__label__1":
        return "negative"
    else:
        return "neutral"

df_full["sentiment"] = df_full["raw_label"].apply(map_fasttext_label)

# 5. Map to emotions

In [5]:
def sentiment_to_emotion(sentiment, text):
    text = text.lower()
    
    if sentiment == "positive":
        if any(w in text for w in ["wow", "amazing", "incredible"]):
            return "surprise"
        return "joy"
    
    elif sentiment == "negative":
        if any(w in text for w in ["angry", "hate"]):
            return "anger"
        elif any(w in text for w in ["sad", "disappointed"]):
            return "sadness"
        elif any(w in text for w in ["fear", "scared"]):
            return "fear"
        else:
            return "anger"
    
    return "neutral"

df_full["label"] = df_full.apply(lambda x: sentiment_to_emotion(x["sentiment"], x["text"]), axis=1)

# 6. Prepare labels and sample data

In [6]:
labels = ["joy", "anger", "sadness", "fear", "surprise", "neutral"]
SAMPLE_SIZE = 150000 

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
df_full["label_id"] = df_full["label"].map(label2id)

# Stratified sampling
print(f"\nSampling {SAMPLE_SIZE} samples...")
df_sampled = df_full.groupby('label_id', group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), max(1, SAMPLE_SIZE // len(labels))), 
        random_state=42
    )
)

if len(df_sampled) < SAMPLE_SIZE:
    remaining = SAMPLE_SIZE - len(df_sampled)
    remaining_samples = df_full[~df_full.index.isin(df_sampled.index)].sample(
        n=min(remaining, len(df_full) - len(df_sampled)), 
        random_state=42
    )
    df_sampled = pd.concat([df_sampled, remaining_samples])
elif len(df_sampled) > SAMPLE_SIZE:
    df_sampled = df_sampled.sample(n=SAMPLE_SIZE, random_state=42)

print(f"Original: {len(df_full)}, Sampled: {len(df_sampled)}")
print("\nClass distribution:")
print(df_sampled['label'].value_counts())

df = df_sampled


Sampling 150000 samples...
Original: 400000, Sampled: 150000

Class distribution:
label
joy         59926
anger       58588
sadness     18279
surprise    12259
fear          948
Name: count, dtype: int64


# 7. Train/Validation Split

In [7]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42
)

print(f"Training: {len(train_df)}, Validation: {len(val_df)}")

Training: 120000, Validation: 30000


# 8. Create Hugging Face datasets

In [8]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# 9. Load model and tokenizer

In [9]:
model_name = "j-hartmann/emotion-english-distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([7]) vs model:torch.Size([6])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([7, 768]) vs model:torch.Size([6, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


# 10. Tokenize

In [10]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64,
    )

print("Tokenizing...")
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])

train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset = val_dataset.rename_column("label_id", "labels")

Tokenizing...


model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

# 11. Class weights

In [11]:
unique_labels = np.unique(df["label_id"])
class_weights_present = compute_class_weight(
    class_weight="balanced",
    classes=unique_labels,
    y=df["label_id"]
)

full_class_weights = np.ones(len(labels), dtype=np.float32)
for i, label_id in enumerate(unique_labels):
    full_class_weights[label_id] = class_weights_present[i]

class_weights = torch.tensor(full_class_weights, dtype=torch.float)
print(f"Class weights: {class_weights}")

Class weights: tensor([ 0.5006,  0.5121,  1.6412, 31.6456,  2.4472,  1.0000])


# 12. Custom trainer

In [12]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Handle DataParallel wrapper
        if hasattr(model, 'module'):
            # If model is wrapped in DataParallel, get the underlying module
            device = model.module.device if hasattr(model.module, 'device') else next(model.parameters()).device
        else:
            device = next(model.parameters()).device
        
        # Move class weights to the correct device
        class_weights_device = class_weights.to(device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_device)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# 13. Metrics

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)
    
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# 14. Training arguments (optimized for Kaggle T4x2)

In [14]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,  
    save_strategy="epoch",
    logging_steps=100,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,  # Mixed precision
    dataloader_num_workers=2,
    gradient_accumulation_steps=1,
    optim="adamw_torch_fused",
    report_to="none",
    save_total_limit=2
)

# 15. Initialize trainer

In [15]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 16. Train

In [16]:
print("\n" + "="*50)
print("Starting training on Kaggle...")
print("="*50 + "\n")

trainer.train()


Starting training on Kaggle...



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.602160,0.575278,0.871733,0.870421,0.871530,0.871733
2,0.485252,0.579844,0.863567,0.863110,0.863573,0.863567


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3750, training_loss=0.5913934015909831, metrics={'train_runtime': 1015.0034, 'train_samples_per_second': 236.452, 'train_steps_per_second': 3.695, 'total_flos': 3974305443840000.0, 'train_loss': 0.5913934015909831, 'epoch': 2.0})

# 17. Evaluate

In [17]:
print("\n" + "="*50)
print("Evaluating...")
print("="*50 + "\n")

eval_results = trainer.evaluate()
print(f"Validation Results:")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall: {eval_results['eval_recall']:.4f}")


Evaluating...



Validation Results:
Accuracy: 0.8716
F1 Score: 0.8703
Precision: 0.8714
Recall: 0.8716


# 18: Save model to Kaggle working directory

In [18]:
model_save_path = "/kaggle/working/emotion_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"\nModel saved to: {model_save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: /kaggle/working/emotion_model


# 19: Download model to locally to the computer

In [19]:
# Create a zip file for easy download
import zipfile
import os

def zip_model(model_path, zip_name="emotion_model.zip"):
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(model_path):
            for file in files:
                zipf.write(os.path.join(root, file), 
                          os.path.relpath(os.path.join(root, file), 
                                        os.path.join(model_path, '..')))
    return zip_name

zip_file = zip_model(model_save_path)
print(f"Model zipped to: {zip_file}")

Model zipped to: emotion_model.zip


# 20. Test the model

In [20]:

def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()
    
    return id2label[predicted_class], confidence

print("\n" + "="*50)
print("Testing model:")
print("="*50)

test_texts = [
    "This product is absolutely amazing! I love it!",
    "I hate this, it's terrible quality",
    "I'm so disappointed with the service",
    "Wow! That was completely unexpected!",
    "I'm really worried about the delivery time"
]

for text in test_texts:
    emotion, confidence = predict_emotion(text)
    print(f"Text: {text}")
    print(f"Prediction: {emotion} ({confidence:.2%} confidence)\n")


Testing model:
Text: This product is absolutely amazing! I love it!
Prediction: surprise (99.71% confidence)

Text: I hate this, it's terrible quality
Prediction: anger (88.37% confidence)

Text: I'm so disappointed with the service
Prediction: sadness (99.63% confidence)

Text: Wow! That was completely unexpected!
Prediction: surprise (98.90% confidence)

Text: I'm really worried about the delivery time
Prediction: fear (87.41% confidence)

